In [ ]:
import gc
import torch

# 先清理一下上一个 notebook 可能残留的显存缓存。
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU cache cleared.')
else:
    print('CUDA not available, skip GPU cache clear.')


# 下一课：CartPole + A2C

前一课我们已经进入了 `Actor-Critic`，理解了：
- Actor 负责输出策略
- Critic 负责评估状态价值
- TD error / advantage 可以作为策略更新信号

这一课我们继续往前走，学习 `A2C`，也就是 `Advantage Actor-Critic`。

这节课的重点是：
- 更明确地用 advantage 来更新 Actor
- 把 actor loss、critic loss、entropy bonus 组合成更标准的训练形式


## 1. A2C 和前一课 Actor-Critic 的关系

你可以把 A2C 理解成：

**把 Actor-Critic 的训练写法整理得更规范、更清楚。**

它通常会明确区分三部分：
- `policy loss`：更新 Actor
- `value loss`：更新 Critic
- `entropy bonus`：鼓励探索

所以这节课不是推翻上一课，而是把上一课的想法变得更标准。


## 2. 什么叫 Advantage

这节课里最关键的量还是：

$A(s, a) = target - V(s)$

你可以把它理解成：
- 这一步动作最终看起来到底比预期好多少
- 如果明显比预期好，就增加这个动作的概率
- 如果比预期差，就降低这个动作的概率

所以 advantage 本质上是在回答：

**“刚才这个动作，相对当前状态的平均水平来说，到底值不值得鼓励？”**


In [ ]:
import random
import warnings

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=3, suppress=True)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
def pick_device():
    if torch.cuda.is_available():
        try:
            _ = torch.zeros(1, device='cuda')
            return torch.device('cuda')
        except Exception as e:
            warnings.warn(f'检测到 CUDA，但当前环境无法真正使用 GPU，已回退到 CPU。原因: {e}')
    return torch.device('cpu')


device = pick_device()
print('当前设备:', device)
if torch.cuda.is_available():
    print('检测到 CUDA 设备:', torch.cuda.get_device_name(0))


In [ ]:
env = gym.make('CartPole-v1')
state, info = env.reset(seed=42)
print('初始状态:', state)
print('状态维度:', env.observation_space.shape[0])
print('动作数量:', env.action_space.n)


In [ ]:
def to_tensor(x, device):
    return torch.tensor(x, dtype=torch.float32, device=device)


class ActorCriticNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.policy_head = nn.Linear(hidden_dim, action_dim)
        self.value_head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        features = self.shared(x)
        logits = self.policy_head(features)
        value = self.value_head(features)
        return logits, value


## 3. A2C 的三部分损失

A2C 的经典训练目标通常由三部分组成：

1. `policy loss`
   用 advantage 更新策略，让好动作概率变大。

2. `value loss`
   让 value 预测更接近目标值。

3. `entropy bonus`
   保留一点探索，防止策略太早变得死板。

最后我们会把这三部分合起来训练。


In [ ]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

model = ActorCriticNet(state_dim, action_dim, hidden_dim=128).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4)

gamma = 0.99
episodes = 500
max_steps = 500
value_coef = 0.5
entropy_coef = 0.01

episode_rewards = []
policy_loss_history = []
value_loss_history = []
entropy_history = []

for episode in range(episodes):
    state, info = env.reset()
    log_probs = []
    values = []
    rewards = []
    entropies = []
    total_reward = 0.0

    for step in range(max_steps):
        state_tensor = to_tensor(state, device).unsqueeze(0)
        logits, value = model(state_tensor)

        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()

        next_state, reward, terminated, truncated, info = env.step(int(action.item()))
        done = terminated or truncated

        log_probs.append(log_prob)
        values.append(value.squeeze(1))
        rewards.append(reward)
        entropies.append(entropy)
        total_reward += reward
        state = next_state

        if done:
            break

    # 计算每一步的 return，作为 value 学习的目标
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32, device=device)

    log_probs = torch.stack(log_probs).squeeze(-1)
    values = torch.cat(values, dim=0)
    entropies = torch.stack(entropies).squeeze(-1)

    # advantage 告诉 Actor：这一动作比当前状态的基准值好多少
    advantages = returns - values.detach()

    policy_loss = -(log_probs * advantages).mean()
    value_loss = nn.functional.mse_loss(values, returns)
    entropy_bonus = entropies.mean()

    # 总损失 = 策略损失 + 价值损失 - 熵奖励
    total_loss = policy_loss + value_coef * value_loss - entropy_coef * entropy_bonus

    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
    optimizer.step()

    episode_rewards.append(total_reward)
    policy_loss_history.append(float(policy_loss.item()))
    value_loss_history.append(float(value_loss.item()))
    entropy_history.append(float(entropy_bonus.item()))

print('训练完成。')
print('最后 20 轮平均 reward:', round(float(np.mean(episode_rewards[-20:])), 2))


## 4. 看训练曲线

这一课你会同时看到：
- reward
- policy loss
- value loss
- entropy

这有助于你理解 A2C 不是只靠一个 loss，而是多部分一起协作。


In [ ]:
window = 10
smoothed_rewards = np.convolve(episode_rewards, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(episode_rewards, color='#1f77b4', alpha=0.5, label='raw')
axes[0, 0].plot(range(window - 1, len(episode_rewards)), smoothed_rewards, color='#d62728', label='smoothed')
axes[0, 0].set_title('每轮 reward')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].legend()

axes[0, 1].plot(policy_loss_history, color='#2ca02c', alpha=0.7)
axes[0, 1].set_title('Policy Loss')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Loss')

axes[1, 0].plot(value_loss_history, color='#ff7f0e', alpha=0.7)
axes[1, 0].set_title('Value Loss')
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('Loss')

axes[1, 1].plot(entropy_history, color='#9467bd', alpha=0.7)
axes[1, 1].set_title('Entropy')
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Entropy')

plt.tight_layout()
plt.show()


## 5. 多次测试平均表现

测试时我们不再采样，而是直接选概率最大的动作，方便看当前策略到底有多稳。


In [ ]:
test_env = gym.make('CartPole-v1')
test_rewards = []

model.eval()
with torch.no_grad():
    for episode_idx in range(5):
        state, info = test_env.reset(seed=123 + episode_idx)
        test_reward = 0.0

        for step in range(500):
            state_tensor = to_tensor(state, device).unsqueeze(0)
            logits, value = model(state_tensor)
            action = int(torch.argmax(logits, dim=1).item())

            state, reward, terminated, truncated, info = test_env.step(action)
            test_reward += reward
            if terminated or truncated:
                break

        test_rewards.append(test_reward)

print('测试 rewards:', test_rewards)
print('测试平均 reward:', round(float(np.mean(test_rewards)), 2))
test_env.close()


## 6. 这节课最该记住什么

如果你想抓住这节课最核心的主线，就记住：

- `policy loss` 让好动作概率上升
- `value loss` 让 Critic 的价值估计更准
- `entropy bonus` 让策略别太早变死板

这三部分一起，构成了一个更标准的 `Advantage Actor-Critic` 训练框架。


## 7. 下一课最自然学什么

学完 A2C 后，最自然的下一步通常是：
- `PPO`

因为 PPO 可以看作是在策略梯度 / Actor-Critic 路线上的一个非常重要、非常实用的升级版。
